# SrOH DC red MOT — full SSE (GPU) with `molmot`

A GPU stochastic-Schrödinger-equation (SSE / MCWF quantum-jump) capture
simulation of SrOH molecules entering a DC red MOT, built with the
[`molmot`](https://github.com/dslmd/molmot) package.

This is the Python/`molmot` analogue of Christian Hallas's Julia notebook
`QuantumSimulations.jl/examples/SrOH_DC_redMOT/SrOH_DC_redMOT.ipynb`:
it launches an **ensemble of independent quantum trajectories in parallel on
the GPU** (one trajectory per CUDA thread-block — the same parallelism the
Julia code gets from its multi-worker cluster) and tracks whether molecules
are captured, plus how many photons they scatter.

## Requirements
- A CUDA GPU + CuPy.  Install molmot with the CUDA extra:
  ```bash
  git clone https://github.com/dslmd/molmot.git && cd molmot
  pip install -e ".[cuda]"
  ```
- Run this notebook from the repo root (so `julia_sim/` is found).

## Physics matched to the Hallas SrOH_DC_redMOT example
- SrOH, 12 ground + 4 excited states, $\lambda=687$ nm, $\Gamma=2\pi\times6.4$ MHz, $m=105$ u.
- 4 laser channels, detunings $[-9.0,-6.2,-19.8,-2.9]$ MHz, $\sigma^+/\sigma^-$.
- $I_\mathrm{sat}=\pi h c\Gamma/(3\lambda^3)$, 50 mW/beam, 10 mm beam radius.
- Anti-Helmholtz gradient $B'=8.8$ G/cm.

> **Note on the solver:** molmot's SSE uses the *direct* MCWF jump method
> (jump when $1-\lVert\psi\rVert^2$ exceeds a uniform threshold) and a
> fixed-step RK4 integrator.  Christian's code uses adaptive DP5.  Results
> are statistically equivalent; step size / tolerances may need tuning.

## 1. Setup and GPU check

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt

# Make sure molmot is importable and julia_sim/ is locatable.
DATA_DIR = 'julia_sim'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join('..', 'julia_sim')
assert os.path.isdir(DATA_DIR), 'Run from the molmot repo root (julia_sim/ not found)'

from molmot_cuda.stochastic_cuda import HAS_CUPY, gpu_info
print('CuPy / CUDA available:', HAS_CUPY)
if HAS_CUPY:
    info = gpu_info()
else:
    print('No GPU detected. This notebook needs a CUDA GPU + CuPy for the ensemble run.')

## 2. Load SrOH molecular data

In [ ]:
from molmot.molecules.sroh import load_sroh_from_julia
from molmot.constants import h, c

mol = load_sroh_from_julia(DATA_DIR, verbose=False)
print(f'{mol.n_ground} ground + {mol.n_excited} excited = {mol.n_states} states')
print(f'lambda = {mol.wavelength*1e9:.0f} nm, Gamma/2pi = {mol.Gamma/2/np.pi/1e6:.1f} MHz, '
      f'mass = {mol.mass/1.66053906660e-27:.0f} u')

## 3. Laser & MOT configuration

Four unique frequency channels.  Channels 1,2 address the lower ground
manifold (`states[end]-states[1]`), channels 3,4 the upper manifold
(`states[end]-states[10]`) — the same references as the Julia notebook.
Each channel is duplicated in the original 8-component scheme; the two
copies average to ~2× intensity, so the per-channel saturation is doubled.

In [ ]:
LAMBDA_M      = 687e-9
BEAM_RADIUS_M = 10e-3
TOTAL_POWER_W = 50e-3
DETUNINGS_MHZ = np.array([-9.0, -6.2, -19.8, -2.9])
POWER_FRAC    = np.array([0.0095, 0.0195, 0.4095, 0.0615])
POLS_Q        = [2, 0, 2, 0]            # sigma+, sigma-, sigma+, sigma-
B_GRAD_TM     = 8.8 / 100.0            # 8.8 G/cm -> T/m

E = mol.energies
Ee = E[mol.n_states - 1]
base = [Ee - E[0], Ee - E[0], Ee - E[9], Ee - E[9]]
freqs = [base[i] + DETUNINGS_MHZ[i] * 1e6 for i in range(4)]

Isat = np.pi * h * c * mol.Gamma / (3 * LAMBDA_M**3)
s0 = 2.0 * POWER_FRAC * TOTAL_POWER_W * (2.0 / (np.pi * BEAM_RADIUS_M**2)) / Isat

beam_pairs = [dict(freq=freqs[i], q_fwd=POLS_Q[i], q_bwd=2 - POLS_Q[i], s0=float(s0[i]))
              for i in range(4)]
for i, bp in enumerate(beam_pairs):
    pol = ['sigma-', 'pi', 'sigma+'][bp['q_fwd']]
    print(f"ch{i+1}: detuning {DETUNINGS_MHZ[i]:+6.1f} MHz, s0={bp['s0']:7.3f}, {pol}")
print(f"I_sat = {Isat*1e-1:.2f} mW/cm^2, B' = 8.8 G/cm, beam radius = 10 mm")

## 4. Build the SSE problem

In [ ]:
from molmot.obe.stochastic import SSEProblem

prob = SSEProblem(mol, beam_pairs, B_gradient=B_GRAD_TM,
                  beam_radius=BEAM_RADIUS_M, add_spontaneous_kick=True)
print('SSEProblem ready. state vector size =', prob.state_size)

## 5. Single-trajectory test

Diagonal approach, like the Julia notebook's test cell:
$r_0=[25,25,0]/\sqrt2$ mm, $v_0=[-11,-11,0]/\sqrt2$ m/s, 60 ms, escape at 20 mm.

The time step must resolve the laser dynamics; `dt = 0.01/Gamma` is the
default.  A 60 ms trajectory is ~$2.4\times10^8$ steps — fine on a GPU, but
set `T_MAX` smaller first if you just want a quick check.

In [ ]:
from molmot_cuda.stochastic_cuda import SSESolverCUDA

R0 = np.array([25e-3, 25e-3, 0.0]) / np.sqrt(2)
V0 = np.array([-11.0, -11.0, 0.0]) / np.sqrt(2)
T_MAX    = 60e-3
R_ESCAPE = 20e-3

solver = SSESolverCUDA(prob)
t0 = time.time()
res = solver.run(R0, V0, t_max=T_MAX, save_every=200000, rng_seed=1,
                 r_escape=R_ESCAPE, renorm_interval=2000)
print(f'wall = {time.time()-t0:.1f} s')
print(f'photons scattered = {res.photons_scattered:,}')
print(f'final |r| = {np.linalg.norm(res.positions[-1])*1e3:.2f} mm, '
      f'|v| = {np.linalg.norm(res.velocities[-1]):.3f} m/s')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
t_ms = res.times * 1e3
for k, lbl in enumerate('xyz'):
    ax1.plot(t_ms, res.positions[:, k]*1e3, label=lbl)
    ax2.plot(t_ms, res.velocities[:, k], label='v'+lbl)
for y in (-20, 20):
    ax1.axhline(y, ls='--', color='grey')
ax1.set_ylabel('position (mm)'); ax1.legend(); ax1.grid(alpha=.3)
ax1.set_title('SrOH DC red MOT — single SSE trajectory')
ax2.set_ylabel('velocity (m/s)'); ax2.set_xlabel('t (ms)')
ax2.axhline(0, ls=':', color='grey'); ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. Ensemble run (the main event)

All trajectories launch in parallel on the GPU.  Christian's notebook used
~110; pick `N` to suit your GPU.  We report the mean photon count and the
capture fraction (final $|r|<20$ mm).

In [ ]:
N = 110
t0 = time.time()
results = solver.run_ensemble(
    n_particles=N,
    r0_sampler=lambda: R0.copy(),
    v0_sampler=lambda: V0.copy(),
    t_max=T_MAX, save_every=200000, rng_seed=1,
    r_escape=R_ESCAPE, renorm_interval=2000,
)
print(f'{N} trajectories in {time.time()-t0:.1f} s')

photons  = np.array([r.photons_scattered for r in results], float)
r_final  = np.array([np.linalg.norm(r.positions[-1]) for r in results])
captured = r_final < R_ESCAPE
print(f'mean photons scattered : {photons.mean():,.0f}  (std {photons.std():,.0f})')
print(f'notebook reference      : ~115,165')
print(f'capture fraction        : {captured.mean()*100:.0f}%  ({captured.sum()}/{N})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# (a) all position-magnitude traces
ax = axes[0]
for r in results:
    ax.plot(r.times*1e3, np.linalg.norm(r.positions, axis=1)*1e3,
            lw=0.5, alpha=0.3, color='C0')
ax.axhline(20, ls='--', color='grey')
ax.set_xlabel('t (ms)'); ax.set_ylabel('|r| (mm)'); ax.set_title('Trajectories')
ax.grid(alpha=.3)
# (b) photon histogram
ax = axes[1]
ax.hist(photons, bins=20, color='C1', alpha=0.8)
ax.axvline(115165, ls='--', color='k', label='notebook mean')
ax.set_xlabel('photons scattered'); ax.set_ylabel('count')
ax.set_title('Photon distribution'); ax.legend(); ax.grid(alpha=.3)
# (c) final radius
ax = axes[2]
ax.hist(r_final*1e3, bins=20, color='C2', alpha=0.8)
ax.axvline(20, ls='--', color='grey', label='escape')
ax.set_xlabel('final |r| (mm)'); ax.set_ylabel('count')
ax.set_title(f'Capture: {captured.mean()*100:.0f}%'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. (Optional) Capture-velocity scan

Like the Julia notebook's purpose: scan the initial inward speed and find the
largest speed that is still captured.  For each speed we run a small ensemble
and record the capture fraction.  Increase `n_per_speed` for smoother curves.

In [ ]:
speeds = np.array([3.0, 6.0, 9.0, 12.0, 15.0, 18.0])   # m/s, inward along xy-diagonal
n_per_speed = 16
cap_frac = []
for sp in speeds:
    v0 = -np.array([1.0, 1.0, 0.0]) / np.sqrt(2) * sp
    res_e = solver.run_ensemble(
        n_particles=n_per_speed,
        r0_sampler=lambda: R0.copy(),
        v0_sampler=lambda v0=v0: v0.copy(),
        t_max=T_MAX, save_every=400000, rng_seed=10,
        r_escape=R_ESCAPE, renorm_interval=2000,
    )
    rf = np.array([np.linalg.norm(r.positions[-1]) for r in res_e])
    frac = (rf < R_ESCAPE).mean()
    cap_frac.append(frac)
    print(f'v0 = {sp:5.1f} m/s -> capture fraction {frac*100:3.0f}%')

cap_frac = np.array(cap_frac)
plt.figure(figsize=(6, 4))
plt.plot(speeds, cap_frac*100, 'o-')
plt.xlabel('initial inward speed (m/s)'); plt.ylabel('capture fraction (%)')
plt.title('SrOH DC red MOT capture vs speed (SSE)'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

---
### Notes
- **Performance.** GPU time scales with `t_max` × `dt`⁻¹ and is largely
  independent of `N` until you saturate the GPU's blocks.  If a full 60 ms
  run is slow, prototype with `T_MAX = 5e-3` and a coarser `save_every`.
- **CPU fallback.** Everything works with `molmot.obe.stochastic.SSESolver`
  (drop-in, same API) but a single 60 ms trajectory is ~75 h in pure Python —
  use the GPU for anything beyond a short sanity check.
- **Fidelity.** The 4-channel scheme here is the incoherent-average of the
  notebook's 8 components (duplicate copies with random phases).  For a
  stricter match, extend `beam_pairs` to 8 entries and add per-beam random
  phases in the field construction.